In [ ]:
import psutil
from functools import partial
import sys
# !{sys.executable} --version
# !{sys.executable} -m pip install shap --upgrade 
import joblib
import time
from shutil import copy
import numpy as np
import pandas as pd
#import tensorflow as tf
import os
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
import pickle
# import xgboost as xgb

from glob import glob
# import psi4
# from helper_CC_ML_spacial import *

import pyscf
from pyscf import gto, scf, mcscf, cc

import ffsim
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.primitives import StatevectorSampler, BitArray
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

from qiskit_addon_sqd.fermion import SCIResult, diagonalize_fermionic_hamiltonian
from qiskit_addon_sqd.counts import bit_array_to_arrays


from ansatzmap import get_zigzag_physical_layout
from math import comb
from tqdm import tqdm

from DDLUCJ import DDLUCJ, GrabAmps


In [ ]:
BasisDirs=glob('data/*')

In [ ]:
energyDF=pd.read_csv("../../../classical/energies.csv",index_col=0)

In [ ]:
energyDF

In [ ]:
moldf = pd.read_csv('molecules.csv')
activespacedf = pd.read_csv("active_spaces.csv")

In [ ]:
BasisSets = ['STO-3G','cc-pVDZ','aug-cc-pVDZ']

In [ ]:
# os.mkdir('energies')

In [ ]:
postprocessed = []
for i in tqdm(sorted(glob("./jobids/*txt")),desc='Running'):
    i.split("/")[-1].replace('.txt','').split('_')
    with open(i,'r') as f:
        name,basis,k,L,JobID = [i.strip() for i in f.readlines()]
    print(name,basis,k,L,JobID)
    moldict = moldf[moldf['molecule']==name]

    n_electrons=moldict['n_electrons'].values[0]
    num_orbitals=moldict['num_orbitals'].values[0]
    xyzname = moldict['mol_filename'].values[0]
    
    pathxyz = os.path.join(os.path.expanduser("~"),"DDLUCJ/classical/structures/",xyzname)



    JobPath = f"./jobids/{name}_LUCJ_L{L}_{basis}_{k}.txt" 
    EnergyPath = f"./energies/{name}_LUCJ_L{L}_{basis}_{k}.txt" 

    # if os.path.exists(JobPath)==True and os.path.exists(EnergyPath)==False:
    print(f"Running {name}_LUCJ_L{L}_{basis}_{k}")
    # run(pathxyz,name,basis,n_electrons,num_orbitals,L,k)
    ampdict = GrabAmps(f"{name}",f"{basis}")
    
    t1, t2 = ampdict[f"{k}"]
    print(comb(num_orbitals,n_electrons//2)**2)

    initDDLUCJ = DDLUCJ(StructurePath=f"{pathxyz}", 
                        BasisSet=f"{basis}", 
                        NElec=int(n_electrons),
                        NOrb=int(num_orbitals),
                        injected=True,
                        t1=t1, 
                        t2=t2,
                        n_reps = int(L),
                        optimization_level=3,
                        temp_dir="./",
                        clean_temp_dir=True,
                        n_jobs=8, 
                        num_batches = 10,
                        max_iterations=5,
                        samples_per_batch=1000,
                        verbose=False)
    
    counts = np.load(f"./counts/{name}_LUCJ_L{L}_{basis}_{k}.npz")
    bitstrings = counts['bitstrings']
    probarr = counts['probarr']
    bitstrings = BitArray.from_bool_array(bitstrings)
    
    result_history, result = initDDLUCJ(postprocess=True,BitArray=bitstrings)     
    new_energy = result.energy + initDDLUCJ.nuclear_repulsion_energy
    EnergyPath = f"./energies/{name}_LUCJ_L{L}_{basis}_{k}_NB10_MI5_spb20_000.txt" 
    with open(EnergyPath,'w') as f:
        new_row={"Basis Set": f"{basis}", "Molecule": f"{name}", "Method": f"LUCJ(L={L})/{k}", "Energy": new_energy}
        for k,v in new_row.items():
            f.write(f"{v}\n")

In [ ]:
num_orbitals,n_electrons